## Prepare your environment

1. **Runtime and compute:** select a Snowflake Notebook on GPU Container Runtime 2.9 with Python 3.12, backed by a GPU compute pool. The query warehouse runs SQL; increasing its size does not increase GPU memory. Choose a pool with enough GPU memory and CPU RAM for the model and graph; there is no validated one-size-fits-all GPU size for this sample.
2. **Local working disk:** provide at least four times the materialized graph-index size in available scratch storage, plus room for retained runs and optional archives. This is starting guidance, not a measured peak or four times compressed source-data size. Confirm that `SCRATCH_DIR` uses the intended supported local disk; the notebook prints available space and actual run usage.
3. **Access and permissions:** package installation needs an approved PyPI route, such as a Snowflake external access integration (EAI). The role needs the privileges below; a successful import does not prove them:

| Action | Required access |
| --- | --- |
| Run notebook | `USAGE` on the GPU compute pool and query warehouse; access to the notebook |
| Install | PyPI access through an approved EAI and `USAGE` on that EAI, where required |
| Read data | `IMPORTED PRIVILEGES` on shared database `SNOWFLAKE_SAMPLE_DATA` (or equivalent inherited access) and query access to `TPCH_SF1` |
| Create views | Permission to create the database/schema and create or replace these views; otherwise have an administrator prepare them and skip the view-setup cell |
| Publish or reload a model | `CREATE MODEL` on the destination schema (or schema ownership) for publication; model `READ`/`USAGE` privileges as appropriate for reload |
| Save predictions (optional) | Create or overwrite the dedicated output table in that schema |
| Archive diagnostics (optional) | Create/write the internal stage and read it for later retrieval |

This notebook uses four TPC-H tables: `CUSTOMER`, `PART`, `ORDERS`, and `LINEITEM`. These are accessed through views created in your configured database/schema over the read-only `SNOWFLAKE_SAMPLE_DATA.TPCH_SF1` data. The LINEITEM view adds a synthetic single-column primary key (`LINEITEM_ID`) since TPC-H uses a composite key (`L_ORDERKEY`, `L_LINENUMBER`).

**Quickstart sample:** by default, the views select 2,000 customers from TPC-H SF1, their orders and line items, and the parts referenced by those line items. This preserves the graph's customer–order–line-item–part relationships. Increase `SAMPLE_CUSTOMERS` only after the small run succeeds.

See [Snowflake GPU runtime 2.9](https://docs.snowflake.com/en/developer-guide/snowflake-ml/container-runtime/releases/gpu/2_9) and [Notebook setup privileges](https://docs.snowflake.com/en/user-guide/ui-snowsight/notebooks-setup) for current platform details.

**Scratch** is local working storage for snapshots, generated rows, model files, predictions, and diagnostics. It may survive a kernel restart if the filesystem remains, but container or machine loss can remove it. Model publication preserves the reusable model; optional archival preserves diagnostics separately.

**Each user is responsible for checking the content of datasets and the applicable licenses and determining if suitable for the intended use**

## 1. Enter your configuration

Fill in your warehouse, database, and schema below. Use ordinary unquoted database and schema identifiers; the notebook validates and normalizes them to uppercase to match Snowflake's unquoted-name resolution. The view-setup cell creates views in your configured database over the read-only `SNOWFLAKE_SAMPLE_DATA.TPCH_SF1`. Views in an owned database are required because Kumo's snapshot uses staged reads, which Snowflake does not allow on shared/imported databases.

The first-run path trains, predicts, and previews locally. Optional diagnostics, plots, durable prediction writes, and archive transfers are disabled by default; enable only the steps you need after the first preview. `MODEL_VERSION` is needed only for reload; `SAVED_ARCHIVE = None` retrieves the archive uploaded by this run.

January 1, 1998 is the configured prediction date. TPC-H SF1 data spans approximately 1992-01-02 to 1998-08-02. After a kernel restart, rerun this configuration and skip installation.

In [ ]:
from datetime import datetime
import re

MODE = "train"  # "train" or "reload"
WAREHOUSE_NAME = "<warehouse>"
DATABASE_NAME = "<database>"
SCHEMA_NAME = "<schema>"

# The view-setup SQL uses unquoted Snowflake identifiers, which resolve in uppercase.
for label, name in (("DATABASE_NAME", DATABASE_NAME), ("SCHEMA_NAME", SCHEMA_NAME)):
    if not isinstance(name, str) or not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_$]*", name):
        raise ValueError(f"{label} must be an unquoted Snowflake identifier; got {name!r}.")
DATABASE_NAME = DATABASE_NAME.upper()
SCHEMA_NAME = SCHEMA_NAME.upper()
SCRATCH_DIR = "/shared/kumo"  # Confirm the backing disk.
MODEL_NAME = "<model>"
MODEL_VERSION = None  # Set the recorded version only for reload.
PREDICTION_TIME = datetime(1998, 1, 1)  # TPC-H data spans ~1992-01-02 to 1998-08-02.
PREDICTION_OUTPUT_TABLE = "<output_table>"
TPCH_SCHEMA = "TPCH_SF1"
SAMPLE_CUSTOMERS = 2000  # Small, relationship-preserving quickstart slice.
ARCHIVE_STAGE = f"@{DATABASE_NAME}.{SCHEMA_NAME}.TRAINING_STAGE/training-results"
SAVED_ARCHIVE = None  # None uses this session's uploaded archive.

# Optional work follows the first prediction preview; enable it explicitly.
INSPECT_DIAGNOSTICS = False
PLOT_LEARNING_CURVES = False
WRITE_PREDICTIONS = False  # True replaces the dedicated output table.
UPLOAD_DIAGNOSTICS = False
DOWNLOAD_DIAGNOSTICS = False
RUN_INDEX = None  # Auto-select this fit's unique new run; set an index to override.
PLAN_INDEX = 0  # Inspect the first available trial plan; choose another if desired.

if TPCH_SCHEMA not in {"TPCH_SF1", "TPCH_SF10"}:
    raise ValueError("Choose TPCH_SF1 or TPCH_SF10.")
if not isinstance(SAMPLE_CUSTOMERS, int) or SAMPLE_CUSTOMERS < 1:
    raise ValueError("SAMPLE_CUSTOMERS must be a positive integer.")

if MODE not in {"train", "reload"}:
    raise ValueError('Choose MODE = "train" or MODE = "reload".')

## 1b. Create views over TPC-H sample data

`SNOWFLAKE_SAMPLE_DATA` is a shared database that does not allow staged exports. This cell creates views in your configured database over a bounded, relationship-preserving slice of four TPC-H SF1 tables so Kumo can unload them to local Parquet. The LINEITEM view adds a synthetic primary key (`LINEITEM_ID`) since TPC-H uses a composite key.

Run this once per environment. Safe to rerun (uses `IF NOT EXISTS` and `OR REPLACE`). The printed row counts describe this run's sample; views do not copy source data.

In [ ]:
from snowflake.snowpark.context import get_active_session
from time import perf_counter

view_setup_started = perf_counter()
session = get_active_session()

# Create writable database with views over read-only SNOWFLAKE_SAMPLE_DATA.
# Staged reads require an owned database; SNOWFLAKE_SAMPLE_DATA is shared and read-only.
session.sql(f"CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}").collect()
session.sql(f"CREATE SCHEMA IF NOT EXISTS {DATABASE_NAME}.{SCHEMA_NAME}").collect()

session.sql(f"""
CREATE OR REPLACE VIEW {DATABASE_NAME}.{SCHEMA_NAME}.CUSTOMER AS
SELECT C_CUSTKEY, C_NAME, C_NATIONKEY, C_MKTSEGMENT, C_ACCTBAL
FROM SNOWFLAKE_SAMPLE_DATA.{TPCH_SCHEMA}.CUSTOMER
WHERE C_CUSTKEY <= {SAMPLE_CUSTOMERS}
""").collect()

session.sql(f"""
CREATE OR REPLACE VIEW {DATABASE_NAME}.{SCHEMA_NAME}.ORDERS AS
SELECT O_ORDERKEY, O_CUSTKEY, O_ORDERDATE, O_ORDERSTATUS, O_ORDERPRIORITY, O_TOTALPRICE, O_SHIPPRIORITY
FROM SNOWFLAKE_SAMPLE_DATA.{TPCH_SCHEMA}.ORDERS
WHERE O_CUSTKEY <= {SAMPLE_CUSTOMERS}
""").collect()

# LINEITEM needs a synthetic single-column primary key (TPC-H uses composite L_ORDERKEY + L_LINENUMBER).
session.sql(f"""
CREATE OR REPLACE VIEW {DATABASE_NAME}.{SCHEMA_NAME}.LINEITEM AS
SELECT
    (L_ORDERKEY * 100 + L_LINENUMBER) AS LINEITEM_ID,
    L_ORDERKEY, L_PARTKEY, L_LINENUMBER, L_QUANTITY,
    L_EXTENDEDPRICE, L_DISCOUNT, L_TAX, L_SHIPDATE,
    L_RETURNFLAG, L_LINESTATUS, L_SHIPMODE
FROM SNOWFLAKE_SAMPLE_DATA.{TPCH_SCHEMA}.LINEITEM
WHERE L_ORDERKEY IN (
    SELECT O_ORDERKEY FROM {DATABASE_NAME}.{SCHEMA_NAME}.ORDERS
)
""").collect()

session.sql(f"""
CREATE OR REPLACE VIEW {DATABASE_NAME}.{SCHEMA_NAME}.PART AS
SELECT P_PARTKEY, P_NAME, P_MFGR, P_BRAND, P_TYPE, P_SIZE, P_CONTAINER, P_RETAILPRICE
FROM SNOWFLAKE_SAMPLE_DATA.{TPCH_SCHEMA}.PART
WHERE P_PARTKEY IN (
    SELECT L_PARTKEY FROM {DATABASE_NAME}.{SCHEMA_NAME}.LINEITEM
)
""").collect()

for table_name in ("CUSTOMER", "ORDERS", "LINEITEM", "PART"):
    count = session.sql(
        f"SELECT COUNT(*) FROM {DATABASE_NAME}.{SCHEMA_NAME}.{table_name}"
    ).collect()[0][0]
    print(f"{table_name} rows: {count:,}")
print(f"View setup and row counts: {perf_counter() - view_setup_started:.1f} seconds")

## 2. Install Kumo Engine from PyPI

Run this installation cell once. **Then restart the kernel, rerun configuration, skip installation, and continue at imports.** Do not use Run All across the restart boundary. Installation uses the active interpreter.

In [ ]:
import subprocess
import sys

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f"Select Python 3.12; found {sys.version.split()[0]}.")

subprocess.check_call([sys.executable, "-m", "pip", "install", "kumo-engine[snowflake]"])
print("Restart the kernel, rerun configuration, then continue with imports.")

<a id="runtime"></a>
## Import and configure scratch

After each kernel restart, run configuration and these cells. They do not depend on variables created during installation. Configure scratch before constructing any Kumo table, graph, or trainer.

In [ ]:
import json
import shutil
import tempfile
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4
import pandas as pd
from IPython.display import display

import kumo.engine as kumo
from kumoapi.typing import Dtype, Stype
from kumoapi.graph import Edge
from kumoapi.jobs import PredictionArtifactType, WriteMode

In [ ]:
# Create and test local scratch, configure Kumo to use it, and verify GPU access.
scratch_root = Path(SCRATCH_DIR).expanduser().resolve()
scratch_root.mkdir(parents=True, exist_ok=True)
with tempfile.TemporaryFile(dir=scratch_root) as probe:
    probe.write(b"kumo scratch check")
    probe.flush()
kumo.configure(scratch_dir=str(scratch_root))
import sys
print("Python:", sys.version.split()[0])
print("Kumo version:", kumo.__version__)
print("GA package:", version("kumo-engine"))
print("Scratch:", scratch_root)
print("Available GiB:", round(shutil.disk_usage(scratch_root).free / 2**30, 1))

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select the matching GPU notebook runtime and compute pool.")
for index in range(torch.cuda.device_count()):
    device = torch.cuda.get_device_properties(index)
    print("GPU:", device.name, "VRAM GiB:", round(device.total_memory / 2**30, 1))

# Measure this notebook's actual core workflow and local scratch checkpoints.
from time import perf_counter
workflow_started_at = perf_counter()
workflow_measurements = []

def scratch_usage_bytes():
    return sum(path.stat().st_size for path in scratch_root.rglob("*") if path.is_file())

def record_workflow_step(name, started_at):
    elapsed = perf_counter() - started_at
    used = scratch_usage_bytes()
    free = shutil.disk_usage(scratch_root).free
    workflow_measurements.append((name, elapsed, used, free))
    print(f"{name}: {elapsed:.1f} seconds; scratch {used / 2**30:.2f} GiB; free {free / 2**30:.2f} GiB")

scratch_at_start = scratch_usage_bytes()
free_at_start = shutil.disk_usage(scratch_root).free

The printed version identifies the installed build. The probe checks write access and free space at this path; it does not establish that the path is on supported local storage. Confirm the backing disk and capacity before materializing data.

## Connect and inspect the source

`from_spcs_env()` uses runtime-provided account and host settings and the mounted OAuth token. Never copy credentials into this notebook. A single connector reads and writes through your configured database/schema. Confirm all four table views appear.

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

for name, value in {"warehouse": WAREHOUSE_NAME, "database": DATABASE_NAME,
                    "schema": SCHEMA_NAME}.items():
    if not value or "<" in value:
        raise ValueError(f"Set your {name} in configuration.")

connector = kumo.SnowflakeConnector.from_spcs_env(
    warehouse=WAREHOUSE_NAME, database=DATABASE_NAME, schema_name=SCHEMA_NAME,
)
print("Tables:", connector.table_names())
display(pd.DataFrame([
    {"column": column.name, "physical_type": column.dtype, "semantic_type": column.stype}
    for column in connector["ORDERS"].columns
]))

## Describe the four tables

A primary key identifies a row. A physical type describes its stored representation, such as a date; a semantic type describes its modeling role, such as a category or numerical value. `with_column()` returns an updated definition. Creating definitions can inspect or sample source data.

The customer and part definitions select the columns used in this example.

In [ ]:
customer = (
    kumo.Table.from_source_table(
        connector["CUSTOMER"],
        ["C_CUSTKEY", "C_NATIONKEY", "C_MKTSEGMENT", "C_ACCTBAL"],
        primary_key="C_CUSTKEY",
    )
    .with_column("C_NATIONKEY", stype=Stype.categorical)
    .with_column("C_MKTSEGMENT", stype=Stype.categorical)
    .with_column("C_ACCTBAL", stype=Stype.numerical)
)

part = (
    kumo.Table.from_source_table(
        connector["PART"],
        ["P_PARTKEY", "P_NAME", "P_BRAND", "P_TYPE",
         "P_SIZE", "P_CONTAINER", "P_RETAILPRICE"],
        primary_key="P_PARTKEY",
    )
    .with_column("P_NAME", stype=Stype.text)
    .with_column("P_BRAND", stype=Stype.categorical)
    .with_column("P_TYPE", stype=Stype.categorical)
    .with_column("P_SIZE", stype=Stype.numerical)
    .with_column("P_CONTAINER", stype=Stype.categorical)
    .with_column("P_RETAILPRICE", stype=Stype.numerical)
)

Event timestamps let Kumo separate information available before a prediction boundary from future outcomes. Using future information as an input causes temporal leakage.

This sample uses `O_ORDERDATE` for orders and `L_SHIPDATE` as the time column for line items. Verify that these timestamps represent appropriate availability boundaries for your use case before adapting these definitions.

In [ ]:
orders = (
    kumo.Table.from_source_table(
        connector["ORDERS"],
        ["O_ORDERKEY", "O_CUSTKEY", "O_ORDERDATE", "O_ORDERSTATUS",
         "O_ORDERPRIORITY", "O_TOTALPRICE"],
        primary_key="O_ORDERKEY",
        time_column="O_ORDERDATE",
    )
    .with_column("O_CUSTKEY", stype=Stype.ID)
    .with_column("O_ORDERDATE", dtype=Dtype.date)
    .with_column("O_ORDERSTATUS", stype=Stype.categorical)
    .with_column("O_ORDERPRIORITY", stype=Stype.categorical)
    .with_column("O_TOTALPRICE", stype=Stype.numerical)
)

In [ ]:
lineitem = (
    kumo.Table.from_source_table(
        connector["LINEITEM"],
        ["LINEITEM_ID", "L_ORDERKEY", "L_PARTKEY", "L_SHIPDATE",
         "L_QUANTITY", "L_EXTENDEDPRICE", "L_DISCOUNT"],
        primary_key="LINEITEM_ID",
        time_column="L_SHIPDATE",
    )
    .with_column("L_ORDERKEY", stype=Stype.ID)
    .with_column("L_PARTKEY", stype=Stype.ID)
    .with_column("L_SHIPDATE", dtype=Dtype.date)
    .with_column("L_QUANTITY", stype=Stype.numerical)
    .with_column("L_EXTENDEDPRICE", stype=Stype.numerical)
    .with_column("L_DISCOUNT", stype=Stype.numerical)
)

## Connect the tables and materialize a snapshot

A **graph** describes the tables and their relationships. Each arrow below goes from a foreign-key column to the destination table's primary key:

```text
LINEITEM.L_PARTKEY  -> PART.P_PARTKEY
LINEITEM.L_ORDERKEY -> ORDERS.O_ORDERKEY
ORDERS.O_CUSTKEY    -> CUSTOMER.C_CUSTKEY
```

Validation checks the graph definition; it is not a blanket guarantee of source-data quality. A **snapshot** materializes source data into the local graph index used for training and prediction. Snapshot creation reads the tables and consumes local disk. Reuse requires an unchanged, valid retained snapshot; loading a model does not restore a lost index.

In [ ]:
graph = kumo.Graph(
    tables=[customer, part, orders, lineitem],
    edges=[
        Edge(src_table="ORDERS", fkey="O_CUSTKEY", dst_table="CUSTOMER"),
        Edge(src_table="LINEITEM", fkey="L_ORDERKEY", dst_table="ORDERS"),
        Edge(src_table="LINEITEM", fkey="L_PARTKEY", dst_table="PART"),
    ],
)

graph.validate()
print("graph id:", graph.id)

from pathlib import Path

snapshot_started_at = perf_counter()
snapshot = graph.snapshot()
record_workflow_step("Graph snapshot", snapshot_started_at)

graph_size_bytes = sum(
    path.stat().st_size
    for path in snapshot.index_path.rglob("*")
    if path.is_file()
)

print("Snapshot:", snapshot.id)
print(f"graph size: {graph_size_bytes / 1024**3:.2f} GB")
print("index path:", snapshot.index_path)

## Define the prediction question

Predictive Query Language (PQL) is Kumo Engine's declarative syntax for defining a predictive modeling task: it specifies the target, the entity to score, and the data context. This default query predicts whether a customer who placed an order in the previous year will place another order in the next 90 days. The `WHERE` clause selects that active-customer population, and `> 0` makes the target binary.

In [ ]:
# Binary classification: predict whether a customer active in the past year will order again in the next 90 days.
pquery = kumo.PredictiveQuery(
    graph=graph,
    query="""
        PREDICT COUNT(ORDERS.*, 0, 90, days) > 0
        FOR EACH CUSTOMER.C_CUSTKEY
        WHERE COUNT(ORDERS.*, -365, 0, days) > 0
    """,
)

pquery.validate()
print("Task type:", pquery.get_task_type())

### Optional task alternatives

The next two code cells are commented out. To try one, skip or comment out the binary query cell above, uncomment exactly one alternative, and run it before training. Both define `pquery` and use the existing `graph`, `snapshot`, and downstream variables. Train a new model for the selected task; do not reload a binary model with a different query. The link example ranks products associated with orders, not customer-level recommendations.

In [ ]:
# Regression: predict how many orders a customer active in the past year will place in the next 90 days.
# pquery = kumo.PredictiveQuery(
#     graph=graph,
#     query=(
#         "PREDICT COUNT(ORDERS.*, 0, 90, days) "
#         "FOR EACH CUSTOMER.C_CUSTKEY "
#         "WHERE COUNT(ORDERS.*, -365, 0, days) > 0"
#     ),
# )
# pquery.validate()
# print("Task type:", pquery.get_task_type())

In [ ]:
# Link prediction: rank up to five parts that may ship in the next 90 days for an order with recent line items.
# pquery = kumo.PredictiveQuery(
#     graph=graph,
#     query=(
#         "PREDICT LIST_DISTINCT(LINEITEM.L_PARTKEY, 0, 90, days) "
#         "RANK TOP 5 FOR EACH ORDERS.O_ORDERKEY "
#         "WHERE COUNT(LINEITEM.*, -365, 0, days) > 0"
#     ),
# )
# pquery.validate()
# print("Task type:", pquery.get_task_type())

### Generate a Training Table

After your query is validated, generate a training table to use for model fitting. This step runs only for `MODE = "train"`; the reload route skips it.

In [ ]:
if MODE == "train":
    generation_started_at = perf_counter()
    train_table = pquery.generate_training_table(snapshot)
    record_workflow_step("Training rows", generation_started_at)
    print("Training table:", train_table.train_table_path)
else:
    print("Reload route: no training table is generated.")

## Create a Trainer

For `MODE = "train"`, the suggested model plan controls architecture, sampling, and optimization. The model plan is fully customizable; inspect and adjust its settings and validation selection metric for your task before creating `trainer = kumo.Trainer(model_plan)`. Compare like-for-like validation metrics and reserve test metrics for evaluating the selected model.

For `MODE = "reload"`, the training-table and trainer-creation steps are skipped. Load the recorded model name/version from a trusted registry publication with compatible graph and query definitions. Reload restores the trained model and generation settings, not source data or snapshot files.

In [ ]:
if MODE == "train":
    model_plan = pquery.suggest_model_plan(snapshot)
    # Quickstart budget; increase after validating the end-to-end path.
    model_plan.training_job.num_experiments = 1
    model_plan.optimization.max_epochs = 3
    # Create a Trainer with your model plan.
    trainer = kumo.Trainer(model_plan)
    print(model_plan)
    print("Validation selection metric:", getattr(
        model_plan.training_job, "tune_metric", "not exposed by this build"))
else:
    if not MODEL_VERSION or "<" in MODEL_VERSION:
        raise ValueError("Set the exact recorded model version before reloading.")
    trainer = kumo.Trainer.load_from_connector(
        connector,
        location=kumo.ModelRegistryLocation(MODEL_NAME, version=MODEL_VERSION),
    )
    print("Loaded:", MODEL_NAME, MODEL_VERSION)

### Train a Model

After creating a Trainer from the model plan, call `fit()` with the training table and compatible graph snapshot. The fit step also publishes the model to Snowflake.

On the first-run route, this cell trains and publishes the selected model to your configured Snowflake database/schema in the same call. It requires model-publication permissions. Record the returned name and version for future reloads. Running the cell again starts another fit.

For a deliberately local-only experiment, set `model_output = None` in this cell before running it. Retain the complete resulting bundle, including manifests and metadata. You can use `kumo.Trainer.load(bundle_path)` only while all required files remain available.

During fit, the cell hides the two known Optuna notices and the pandas incompatible-dtype FutureWarning. This reduces output noise; it does not repair the underlying pandas assignment. Other warnings remain visible, and the filters are restored after fit.

In [ ]:
if MODE == "train":
    # Capture existing runs immediately before fitting; old runs must not be selected automatically.
    runs_before_fit = {
        path.parent.resolve()
        for path in scratch_root.rglob("auto_train_progress.json")
        if path.is_file()
    }
    fit_started_at = perf_counter()
    model_output = kumo.ModelOutputConfig(
        connector, location=kumo.ModelRegistryLocation(MODEL_NAME),
    )
    import warnings
    from optuna.exceptions import ExperimentalWarning

    # Hide only these known notices during fit; preserve other warnings.
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message=r"`warn_independent_sampling` has been deprecated.*",
            category=FutureWarning,
        )
        warnings.filterwarnings(
            "ignore",
            message=r"Argument ``multivariate`` is an experimental feature.*",
            category=ExperimentalWarning,
        )
        warnings.filterwarnings(
            "ignore",
            message=r"Setting an item of incompatible dtype is deprecated.*",
            category=FutureWarning,
        )
        warnings.filterwarnings(
            "ignore",
            message=r".*torch-scatter.*",
            category=UserWarning,
        )
        training_result = trainer.fit(
            train_table, snapshot=snapshot, export_config=model_output,
        )
    record_workflow_step("Fit and publish", fit_started_at)
    published = training_result.published_model
    if published is not None:
        print("Record registry:", DATABASE_NAME, SCHEMA_NAME)
        print("Record model name/version:", published.name, published.version)
    else:
        print("Local-only model. Retain complete bundle:", training_result.bundle_dir)
else:
    print("Reload route: fit and publication skipped.")

### Inspect the selected result

Review the reported metrics and their stage names. Available metrics vary by task and build; this example does not promise calibration output or a fixed set. A successful fit alone does not establish business usefulness.

The reusable bundle and registry publication preserve model state. They do not archive the complete experiment history or explanations. Continue to batch prediction below; optional diagnostics follow the prediction preview.

In [ ]:
if MODE == "train":
    print("Available metrics:", training_result.metrics)
    print("Reusable bundle:", training_result.bundle_dir)
    for name, path in training_result.artifact_paths.items():
        print(name, path)
else:
    print("Model loaded. Earlier experiment diagnostics require retained scratch or an archive.")

<a id="prediction"></a>
## Generate Batch Predictions

After training a model, generate batch predictions and optionally write them to a Snowflake table. The reload route can instead score with a previously published model.

Check the source date range against `PREDICTION_TIME` before scoring. A chosen date alone does not guarantee the following 90 days of observed outcomes for historical evaluation.

In [ ]:
from snowflake.snowpark.functions import col, min as sf_min, max as sf_max

def quoted_identifier(name):
    return '"' + name.replace('"', '""') + '"'

order_dates = session.table([
    quoted_identifier(name)
    for name in [DATABASE_NAME, SCHEMA_NAME, "ORDERS"]
])
display(order_dates.select(
    sf_min(col('"O_ORDERDATE"')).alias("FIRST_ORDER"),
    sf_max(col('"O_ORDERDATE"')).alias("LAST_ORDER"),
).collect())
print("Configured example boundary:", PREDICTION_TIME)

### Generate the scoring population

For this temporal query, use a timezone-naive timestamp interpreted as UTC. Reuse the fitted trainer's saved training-table generation plan. The generated population contains customers meeting the past-year restriction at that date. This timestamp example does not apply to static queries.

In [ ]:
prediction_table_started_at = perf_counter()
prediction_table = pquery.generate_prediction_table(
    snapshot,
    prediction_time=PREDICTION_TIME,
    plan=trainer.training_table_generation_plan,
)
record_workflow_step("Prediction rows", prediction_table_started_at)

### Score locally, with an optional Snowflake output

**`WRITE_PREDICTIONS = True` authorizes `OVERWRITE`, which replaces the configured destination table's contents.** Choose a dedicated table you own, never a source view, and review the configuration before running this cell. Writing is disabled by default, so the first preview reads local Parquet files; set `WRITE_PREDICTIONS = True` to persist results. Select a different output name to retain an earlier batch.

Inference options follow the active task: binary classification uses a 0.5 threshold, regression uses no extra option, and link prediction requests five ranked products per order. Link scores rank candidates; they are not calibrated probabilities.

In [ ]:
prediction_output = None
if WRITE_PREDICTIONS:
    import re
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREDICTION_OUTPUT_TABLE):
        raise ValueError("Choose a simple dedicated output table name in configuration.")
    source_names = {"CUSTOMER", "PART", "ORDERS", "LINEITEM"}
    if PREDICTION_OUTPUT_TABLE.upper() in source_names:
        raise ValueError("Prediction output must not replace a source table.")
    prediction_output = kumo.PredictionOutputConfig(
        connector=connector, artifact_type=PredictionArtifactType.PREDICTIONS,
        table_name=PREDICTION_OUTPUT_TABLE, write_mode=WriteMode.OVERWRITE,
    )
from kumoapi.task import TaskType

prediction_options = {}
if pquery.get_task_type() == TaskType.BINARY_CLASSIFICATION:
    prediction_options["binary_classification_threshold"] = 0.5
elif pquery.get_task_type() == TaskType.TEMPORAL_LINK_PREDICTION:
    prediction_options["num_classes_to_return"] = 5
prediction_started_at = perf_counter()
prediction_result = trainer.predict(
    prediction_table, snapshot=snapshot,
    output_config=prediction_output, **prediction_options,
)
record_workflow_step("Prediction", prediction_started_at)
print("Rows:", prediction_result.num_rows)
print("Local Parquet files:", prediction_result.files)
if WRITE_PREDICTIONS:
    print("Snowflake output:", DATABASE_NAME, SCHEMA_NAME, PREDICTION_OUTPUT_TABLE)

### Read a bounded preview

Read at most ten rows across the local Parquet files in batches. This avoids collecting the entire result. Parquet readers still need file metadata and decoding buffers. `prediction_result.frame` instead loads all rows into memory, even if followed by `.head()`.

In [ ]:
import pyarrow.parquet as pq

def preview_parquet(files, limit=10):
    frames = []
    remaining = limit
    for path in files:
        if remaining <= 0:
            break
        with pq.ParquetFile(path) as parquet:
            for batch in parquet.iter_batches(batch_size=remaining):
                frames.append(batch.slice(0, remaining).to_pandas())
                remaining -= len(frames[-1])
                if remaining <= 0:
                    break
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

display(preview_parquet(prediction_result.files))

### Measured first-run time and local storage

Run this cell immediately after the prediction preview. It reports elapsed time for the core workflow (from scratch configuration through prediction), measured step times, and local scratch usage at checkpoints. View setup and package installation are timed separately or excluded because installation crosses a kernel restart. The largest observed scratch checkpoint is **not** a true peak or a guaranteed capacity limit; temporary files may grow between measurements. The available-space figure also reflects other files on the same filesystem. Plan headroom using the earlier four-times-graph-size starting guidance, then use these measurements from your own account to size subsequent runs. No universal SF1 benchmark is implied.

In [ ]:
measurement_rows = [
    {
        "step": name,
        "elapsed_seconds": round(elapsed, 1),
        "scratch_gib_after": round(used / 2**30, 2),
        "free_gib_after": round(free / 2**30, 2),
    }
    for name, elapsed, used, free in workflow_measurements
]
display(pd.DataFrame(measurement_rows))

scratch_now = scratch_usage_bytes()
observed_high = max([scratch_at_start, scratch_now] + [row[2] for row in workflow_measurements])
free_now = shutil.disk_usage(scratch_root).free
print(f"Measured core workflow elapsed: {perf_counter() - workflow_started_at:.1f} seconds")
print(f"Scratch at start: {scratch_at_start / 2**30:.2f} GiB")
print(f"Scratch after preview: {scratch_now / 2**30:.2f} GiB")
print(f"Largest observed scratch checkpoint: {observed_high / 2**30:.2f} GiB")
print(f"Free local disk now: {free_now / 2**30:.2f} GiB")
print(f"Graph index size: {graph_size_bytes / 2**30:.2f} GiB")
print("These are run-specific measurements, not a peak-memory or disk guarantee.")

<a id="diagnostics"></a>
## Optional: inspect experiments and explanations

Inspection is disabled by default to keep the first run short; enable it after the prediction preview. Nothing here is required for prediction. Discovery lists retained AutoML directories. The fit cell records existing run directories immediately before training. The notebook automatically selects a run only when exactly one new directory appeared after that fit. It never assumes the latest directory is this model's run.

If selection is ambiguous, set `RUN_INDEX` in configuration to a displayed new-run index and rerun selection. On reload, select a retained run explicitly; loading a registry model does not restore its diagnostics. Set `INSPECT_DIAGNOSTICS = True` to inspect the run. Plotting has its own switch.

In [ ]:
automl_dirs = sorted({
    path.parent.resolve() for path in scratch_root.rglob("auto_train_progress.json")
    if path.is_file()
})
for index, path in enumerate(automl_dirs):
    print(index, path)
if not automl_dirs:
    print("No retained AutoML runs found under configured scratch.")

In [ ]:
automl_dir = None
if INSPECT_DIAGNOSTICS:
    if MODE == "train":
        if "runs_before_fit" not in globals():
            raise RuntimeError("Run the fit cell first to capture existing run directories.")
        candidate_indices = [
            index for index, path in enumerate(automl_dirs)
            if path not in runs_before_fit
        ]
        if RUN_INDEX is None and len(candidate_indices) == 1:
            automl_dir = automl_dirs[candidate_indices[0]]
        elif RUN_INDEX is None:
            print("New run indices:", candidate_indices)
            print("Set RUN_INDEX to the intended new run index and rerun this cell.")
        elif not isinstance(RUN_INDEX, int) or RUN_INDEX not in candidate_indices:
            raise ValueError(
                f"RUN_INDEX must select a new run from {candidate_indices}; "
                "old runs cannot be used for this fit."
            )
        else:
            automl_dir = automl_dirs[RUN_INDEX]
    elif RUN_INDEX is None:
        print("Reload has no current fit. Set RUN_INDEX to a retained run explicitly.")
    elif not isinstance(RUN_INDEX, int) or not 0 <= RUN_INDEX < len(automl_dirs):
        raise ValueError("Choose one of the displayed retained run indices.")
    else:
        automl_dir = automl_dirs[RUN_INDEX]
    if automl_dir is not None:
        print("Selected:", automl_dir)
else:
    print("Diagnostic inspection disabled.")

### Read progress and compare trials

These small readers report missing or unreadable files without blocking prediction. The progress summary shows completed/total trials and each trial's recorded epochs, elapsed seconds, and accelerator. Missing optional values mean they were not recorded.

In [ ]:
def read_json(path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, ValueError) as error:
        print(f"Cannot read {path.name}: {error}")
        return {}

def read_csv(path):
    try:
        return pd.read_csv(path)
    except (OSError, pd.errors.EmptyDataError, pd.errors.ParserError) as error:
        print(f"Cannot read {path.name}: {error}")
        return pd.DataFrame()

def history_frames(progress):
    summaries, curves = [], []
    for trial, details in (progress.get("trial_progress") or {}).items():
        details = details or {}
        summaries.append({"trial": str(trial),
                          "epochs_completed": details.get("num_epochs_completed"),
                          "elapsed_seconds": details.get("elapsed_time"),
                          "accelerator": details.get("accelerator")})
        for epoch, metrics in (details.get("metrics") or {}).items():
            try:
                epoch_number = int(epoch)
            except (TypeError, ValueError):
                print("Skipping unrecognized epoch:", epoch)
                continue
            metrics = metrics or {}
            curves.append({"trial": str(trial), "epoch": epoch_number,
                           **(metrics.get("train_metrics") or {}),
                           **(metrics.get("validation_metrics") or {})})
    summary = pd.DataFrame(summaries)
    history = pd.DataFrame(curves)
    if not history.empty:
        history = history.sort_values(["trial", "epoch"])
    return summary, history

In [ ]:
progress = read_json(automl_dir / "auto_train_progress.json") if automl_dir else {}
trial_summary, learning_curves = history_frames(progress)
print("Completed / total trials:", progress.get("completed_trials", "not recorded"),
      "/", progress.get("total_trials", "not recorded"))
if not trial_summary.empty:
    display(trial_summary)
else:
    print("No trial progress recorded or no run selected.")
if not learning_curves.empty:
    display(learning_curves)
print("Available epoch metrics:", [
    c for c in learning_curves.columns if c not in {"trial", "epoch"}
])

In [ ]:
trial_frames = []
for path in sorted(automl_dir.glob("version_*/val_result.csv")) if automl_dir else []:
    frame = read_csv(path)
    if not frame.empty:
        frame["trial_directory"] = path.parent.name
        trial_frames.append(frame)
if trial_frames:
    display(pd.concat(trial_frames, ignore_index=True))
else:
    print("No completed trial validation results found.")

plan_paths = sorted(automl_dir.glob("version_*/model_plan.json")) if automl_dir else []
for index, path in enumerate(plan_paths):
    print(index, path.relative_to(automl_dir))

The first available trial plan is displayed by default. Change `PLAN_INDEX` in configuration to inspect another. Compare trial validation results using the configured selection metric, not an assumed metric or direction.

In [ ]:
if plan_paths and PLAN_INDEX is not None:
    if not isinstance(PLAN_INDEX, int) or not 0 <= PLAN_INDEX < len(plan_paths):
        raise ValueError("Choose a displayed plan index.")
    display(read_json(plan_paths[PLAN_INDEX]))
else:
    print("No trial plan selected.")

### Plot available learning curves

Select exact metric names printed above. The default tries training and validation loss. Compare how validation changes while training improves. Divergence can suggest overfitting; nearly perfect results may warrant a leakage check. These are diagnostic signals, not proof of a cause. Plotting is optional and does not affect other inspection cells.

In [ ]:
METRIC_COLUMNS = ["train_loss", "val_loss"]
columns = [name for name in METRIC_COLUMNS if name in learning_curves.columns]
if not PLOT_LEARNING_CURVES:
    print("Learning-curve plotting disabled.")
elif learning_curves.empty or not columns:
    print("No matching completed epoch metrics. Choose from the available names.")
else:
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("Matplotlib is unavailable. Use the displayed epoch table instead.")
    else:
        figure, axis = plt.subplots(figsize=(9, 5))
        for trial, rows in learning_curves.groupby("trial", sort=True):
            for column in columns:
                values = pd.to_numeric(rows[column], errors="coerce")
                if values.notna().any():
                    axis.plot(rows["epoch"], values, marker="o", label=f"{trial}: {column}")
        axis.set(xlabel="Epoch", ylabel="Metric", title="Learning curves")
        axis.grid(alpha=0.25)
        if axis.lines:
            axis.legend()
        plt.show()

### Inspect gradient scores and column analysis

Post-fit gradient scoring and column analysis availability depends on the installed GA package, task, and model. Explanations can be disabled through the model plan's `training_job.disable_explain` setting.

Gradient scores describe model sensitivity to input columns, not causal effects. Column analysis groups feature values into cohorts and summarizes predictions and outcomes. Explanation errors may be logged while the model remains usable. Missing scores do not prove a column was excluded from training.

The following cells search beneath the selected run, display the actual artifact paths, and read what exists. They do not use a separate explanation API or require the plotting cell.

In [ ]:
score_paths = sorted(automl_dir.rglob("explain_test/scores.csv")) if automl_dir else []
if not score_paths:
    print("No gradient scores. Check selected run, build, explanation setting, and logs.")
for path in score_paths:
    print("Scores:", path.relative_to(automl_dir))
    scores = read_csv(path)
    if "score" in scores.columns:
        scores = scores.sort_values("score", ascending=False)
    display(scores)

In [ ]:
analysis_paths = sorted(automl_dir.rglob(
    "column_analysis/test/column_analysis.json"
)) if automl_dir else []
if not analysis_paths:
    print("No column analysis found for the selected run.")
for path in analysis_paths:
    print("Column analysis:", path.relative_to(automl_dir))
    analysis = read_json(path)
    display(pd.json_normalize(analysis.get("cohorts") or []))

### Locate other diagnostics

When present, `best_trainer.json` identifies the selected trial; `train_progress.json` records individual trial progress; TensorBoard event files support deeper curve inspection; `failed_trials` contains failure diagnostics; and `analyze` contains optimization analysis. Parameter importance in optimization analysis concerns training settings, not input-column explanations. Not every run emits every artifact type.

In [ ]:
if automl_dir:
    for pattern in ["best_trainer.json", "version_*/train_progress.json",
                    "**/events.out.tfevents.*", "failed_trials/*", "analyze/*"]:
        paths = sorted(automl_dir.glob(pattern))
        print(pattern, ":", len(paths), "matches")
        for path in paths[:20]:
            print(" ", path.relative_to(automl_dir))

## Optional: preserve diagnostics

Archive only the selected AutoML directory to a restricted internal stage. Archives can contain large checkpoints and sensitive diagnostic information. Allow additional local space for the ZIP while the original files remain. This is separate from model publication: an archive is not a registry entry, source-data backup, or snapshot restore mechanism.

Upload is disabled by default. The upload cell creates the configured stage when enabled; set `UPLOAD_DIAGNOSTICS = True` if you need a durable archive. A selected run is required. The unique filename avoids replacing earlier archives. This cell checks upload results and prints the exact path to record. It does not delete scratch.

In [ ]:
uploaded_archive_stage_path = None
if UPLOAD_DIAGNOSTICS and INSPECT_DIAGNOSTICS and automl_dir is not None:
    if not ARCHIVE_STAGE or "<" in ARCHIVE_STAGE or not ARCHIVE_STAGE.startswith("@"):
        raise ValueError("Configure an existing writable internal archive stage.")
    session.sql(
        f"CREATE STAGE IF NOT EXISTS {DATABASE_NAME}.{SCHEMA_NAME}.TRAINING_STAGE"
    ).collect()
    archive_dir = Path(tempfile.mkdtemp(prefix="diagnostic-export-", dir=scratch_root))
    archive_name = f"tpch-{automl_dir.name}-{uuid4().hex}"
    archive_path = shutil.make_archive(str(archive_dir / archive_name), "zip", root_dir=automl_dir)
    results = session.file.put(
        f"file://{archive_path}", ARCHIVE_STAGE, overwrite=False, auto_compress=False,
    )
    for result in results:
        print(result.target, result.status, result.message)
    if not results or any(result.status != "UPLOADED" for result in results):
        raise RuntimeError("Archive upload incomplete. Review results; local files remain.")
    uploaded_archive_stage_path = f"{ARCHIVE_STAGE.rstrip('/')}/{Path(archive_path).name}"
    print("Record archive path:", uploaded_archive_stage_path)
else:
    print("Archive upload skipped: no run selected or upload disabled.")

### Retrieve and inspect an archive

Download is disabled by default. Set `DOWNLOAD_DIAGNOSTICS = True` to retrieve the exact archive path returned by this session's upload. To inspect a previous archive, set `SAVED_ARCHIVE` in configuration to its recorded full stage path. If upload is disabled, a saved path is required.

This cell uses the active Snowpark session and a fresh directory. It lists ZIP contents without extracting or executing them. Locate validation CSVs and plan JSON by their listed paths. Local files last only as long as the environment retains its filesystem.

In [ ]:
archive_to_download = SAVED_ARCHIVE or uploaded_archive_stage_path
if DOWNLOAD_DIAGNOSTICS and archive_to_download:
    from zipfile import ZipFile
    if "<" in archive_to_download or not archive_to_download.endswith(".zip"):
        raise ValueError("Set the exact saved archive stage path in configuration.")
    download_dir = Path(tempfile.mkdtemp(prefix="diagnostic-download-", dir=scratch_root))
    downloads = session.file.get(archive_to_download, str(download_dir))
    downloaded_zip = download_dir / archive_to_download.rsplit("/", 1)[-1]
    if not downloads or not downloaded_zip.is_file():
        raise FileNotFoundError("Archive download failed. Check stage access and path.")
    with ZipFile(downloaded_zip) as archive:
        members = archive.infolist()
        print("Members:", len(members))
        for member in members[:50]:
            print(member.filename, member.file_size, "bytes")
    print("Downloaded:", downloaded_zip)
else:
    print("Archive retrieval skipped: no saved or uploaded archive, or download disabled.")

## Next steps

To score again without retraining, record the model name/version and database/schema, restart or open a new session, and choose `MODE = "reload"`. Reconnect and reconstruct the compatible graph, query, and snapshot before prediction. Retained snapshot files can be reused when valid; a lost snapshot must be materialized again from the source.

For further experiments, inspect the suggested and saved trial plans before changing epoch limits, sampling, or architecture. Compare validation results under consistent splits. Changing the graph, target, or horizon calls for revalidation and may require retraining. Other supported task types, such as regression, forecasting, and link prediction, need their own validated query and task-appropriate evaluation.

**Embeddings are an optional extension**, separate from prediction probabilities. Current local APIs select an embedding artifact through prediction output configuration and run another prediction call. Check the installed GA package's support and output contract before adding that extension; it is intentionally outside this executable classification path.

Keep the published model, diagnostic archive, and source/snapshot lifecycle separate when planning repeat runs.